# Essay Writer Agent

In [47]:
from dotenv import load_dotenv, find_dotenv 

_ = load_dotenv(find_dotenv())

In [48]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List 
import operator 
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver 
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [49]:
# creating an agent state
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

In [50]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="qwen3.8:27b")  

In [51]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

In [52]:
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed: 

------

{content}"""

In [53]:
REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

In [54]:
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""

In [55]:
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""

In [56]:
from pydantic import BaseModel

class Queries(BaseModel):
    queries: List[str]

In [57]:
from tavily import TavilyClient 
import os
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [58]:
# create teh plan node
def plan_node(state:AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state["task"])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

In [59]:
def research_plan_node(state:AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state["task"])])
    content = state['content'] or []
    for q in queries.queries: 
        response = tavily.search(query=q, max_results=3)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}
    

In [60]:
def generation_node(state:AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state["plan"]}"
    )
    messages = [
        SystemMessage(content=WRITER_PROMPT.format(content=content)),
        user_message
    ]
    response = model.invoke(messages)
    return {"draft": response.content, "revision_number": state.get("revision_number", 1) + 1}

In [61]:
def reflection_node(state:AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state["draft"])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [62]:
def research_critique_node(state:AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=3)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [63]:
def should_continue(state:AgentState):
    return END if state.get("revision_number", 1) > state.get("max_revisions", 3) else "reflect"

In [64]:
# intialize a graph 
builder = StateGraph(AgentState)
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [65]:
builder.set_entry_point("planner")

In [66]:
builder.add_conditional_edges(
    "generate",
    should_continue,
    {END: END, "reflect": "reflect"}
)

In [67]:
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

In [68]:
graph = builder.compile(checkpointer=memory)

In [70]:
# use the agent 
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    "task": "what is the difference between langchain and langsmith",
    "max_revisions": 2,
    "revision_number": 1, 
    "content": []}, thread):
    print(s)

{'planner': {'plan': '# Essay Outline: LangChain vs. LangSmith — Understanding the Distinction\n\n---\n\n## I. Introduction\n- **Opening hook:** The rapid growth of LLM-powered applications has created a dense ecosystem of tools, often conflated by name and branding.\n- **Thesis statement:** While LangChain and LangSmith share a common origin and complement each other in practice, they occupy fundamentally different roles in the LLM application lifecycle — one is a *development framework* for building LLM applications, and the other is an *observability and evaluation platform* for monitoring, debugging, and improving them.\n- **Brief roadmap** of what the essay will cover.\n\n> **Note for writer:** Keep the intro tight (2–3 short paragraphs). Avoid diving into technical detail here. The goal is to frame the "confusion" many developers feel and promise clarity.\n\n---\n\n## II. LangChain: The Development Framework\n- **A. Core purpose**\n  - An open-source framework for *composing* LLM